## Connector demo
compatible with 
- EDC v0.10.1, 
- EDC v0.14.0

most examples based on the edc samples: https://github.com/eclipse-edc/Samples

In [ ]:
import requests
import json
import time
from typing import List, Dict
import re
from urllib.parse import unquote
from uuid import uuid4

from dataspace_apis import *

### Demo setup

In [3]:

IS_LOCALHOST_DEPLOYMENT = False

PROVIDER_URL = "https://sage-connector-edc-connector.apps.bst2.paas.psnc.pl"
CONSUMER_URL = "https://consumer-connector-edc-connector.apps.bst2.paas.psnc.pl"
FEDERATED_CATALOG_BASE_URL = "https://federatedcatalog-edc-connector.apps.bst2.paas.psnc.pl"
CONSUMER_BACKEND_URL = "https://consback-edc-connector.apps.bst2.paas.psnc.pl"

# for local:
LOCALHOST = "http://localhost"
CONSUMER_CONTAINER = "http://consumer-connector"
PROVIDER_CONTAINER = "http://provider-connector"
FEDERATED_CATALOG_BASE_CONTAINER = "http://federated-catalog"
CONSUMER_BACKEND_CONTAINER = "http://consumer-backend"

if (IS_LOCALHOST_DEPLOYMENT):
    CONSUMER_URL = LOCALHOST
    PROVIDER_URL = LOCALHOST
    FEDERATED_CATALOG_BASE_URL = LOCALHOST

In [ ]:
PROVIDER_API = f"{PROVIDER_URL}/api"
PROVIDER_CONTROL = f"{PROVIDER_URL}/api/control"
PROVIDER_MANAGEMENT = f"{PROVIDER_URL}/api/management"
PROVIDER_PROTOCOL = f"{PROVIDER_URL}/api/dsp"
PROVIDER_PUBLIC = f"{PROVIDER_URL}/api/public"

CONSUMER_API = f"{CONSUMER_URL}/api"
CONSUMER_CONTROL = f"{CONSUMER_URL}/api/control"
CONSUMER_MANAGEMENT = f"{CONSUMER_URL}/api/management"
CONSUMER_PROTOCOL = f"{CONSUMER_URL}/api/dsp"
CONSUMER_PUBLIC = f"{CONSUMER_URL}/api/public"

CONSUMER_BACKEND_EDR = f"{CONSUMER_BACKEND_URL}/edr-endpoint"
FEDERATED_CATALOG_URL = f"{FEDERATED_CATALOG_BASE_URL}/api/catalog"

if (IS_LOCALHOST_DEPLOYMENT):
    PROVIDER_API = f"{PROVIDER_URL}:19191/api"
    PROVIDER_CONTROL = f"{PROVIDER_URL}:19192/api/control"
    PROVIDER_MANAGEMENT = f"{PROVIDER_URL}:19193/api/management"
    PROVIDER_PROTOCOL = f"{PROVIDER_URL}:19194/api/dsp"
    PROVIDER_PUBLIC = f"{PROVIDER_URL}:19291/api/public"

    CONSUMER_API = f"{CONSUMER_URL}:29191/api"
    CONSUMER_CONTROL = f"{CONSUMER_URL}:29192/api/control"
    CONSUMER_MANAGEMENT = f"{CONSUMER_URL}:29193/api/management"
    CONSUMER_PROTOCOL = f"{CONSUMER_URL}:29194/api/dsp"
    CONSUMER_PUBLIC = f"{CONSUMER_URL}:29291/api/public"

    CONSUMER_BACKEND_EDR = f"{CONSUMER_BACKEND_CONTAINER}:4000/edr-endpoint"
    FEDERATED_CATALOG_URL = f"{FEDERATED_CATALOG_BASE_URL}:9181/api/catalog"

In [5]:
"""
The vars below are useful when working on local deployment
(or at least when no routings are specified for connectors)
Docker containers have their own localhost, which is not the host
machine's localhost.

In some requests there is a 'counterPartyAddress' which contains
localhost. This one will be solved as containers' internal localhost
but not the localhost of the host machine and connectors won't connect
to each other.

Hence we substitute the "localhost" with containers' names (if they
contain "localhost", otherwise urls remain unchanged).
If routings are specified correctly on PaaS, those lines aren't
required.

We leave those vars here anyway, just not to complicate 
any of the requests later in the demo. 
"""

provider_control_internal = PROVIDER_CONTROL.replace(LOCALHOST, PROVIDER_CONTAINER)
provider_public_internal = PROVIDER_PUBLIC.replace(LOCALHOST, PROVIDER_CONTAINER)
provider_protocol_internal = f"{PROVIDER_PROTOCOL}".replace(LOCALHOST, PROVIDER_CONTAINER)

default_headers = {
    "Content-Type": "application/json",
    "x-api-key": "edc",
}

### Conn Check

In [6]:
def check_health(coreApiUrl):
    print(f"{coreApiUrl}/check/health/")
    rp = requests.get(f"{coreApiUrl}/check/health/", headers=default_headers).json()
    print(rp)

check_health(PROVIDER_API)
check_health(CONSUMER_API)
print("They're Alive!")

https://sage-connector-edc-connector.apps.bst2.paas.psnc.pl/api/check/health/
{'componentResults': [{'failure': None, 'component': 'Dataplane Self Registration', 'isHealthy': True}, {'failure': None, 'component': 'BaseRuntime', 'isHealthy': True}], 'isSystemHealthy': True}
https://consumer-connector-edc-connector.apps.bst2.paas.psnc.pl/api/check/health/
{'componentResults': [{'failure': None, 'component': 'Dataplane Self Registration', 'isHealthy': True}, {'failure': None, 'component': 'BaseRuntime', 'isHealthy': True}], 'isSystemHealthy': True}
They're Alive!


### Content reader script 

In [9]:
def extract_name_from_uri(uri: str) -> str:
    """
    Extracts the name from a URI.
    Assumes the name is the last part of the URI after the last slash.
    """
    name = uri.split('/')[-1]
    _hash = ''
    if '?' in name:
        res = re.split(r'\?|\%3F', name, maxsplit=1)
        if len(res) == 1:
            name = res[0]
        elif len(res) == 2:
            name, _hash = res
        _hash = str(hash(_hash))
    return f"{name}{"_" + _hash if _hash != "" else ""}"

In [23]:
def extract_content_type_from_uri(uri: str) -> str:
    """
    Extracts the content type from a URI.
    Assumes the content type is the file extension of the last part of the URI.
    """
    name = uri.split('/')[-1]
    if '.' in name:
        return name.split('.')[-1]
    return "json"

In [ ]:
path = '/Users/matt/Downloads/dcat-dump.jsonld'
URL = 'dcat:landingPage'
IDENTIFIER = 'dct:identifier'
TITLE = 'dct:title'
DESCRIPTION = 'dct:description'
SPATIAL = 'http://purl.org/dc/terms/spatial'

records : Dict = {}

with open(path, 'r') as f:
    records = json.load(f)

policy_id = "test-policy"
policy = create_policy(policy_id, PROVIDER_MANAGEMENT, default_headers, permissions=[])
print(f'Policy uploaded with status code: {policy.status_code}')

print(records["@context"])
for record in records["dcat:dataset"]:
    asset_id = record[IDENTIFIER] or uuid4()
    contract_definition_id = f"cd_{asset_id}"
    url = record[URL]["@id"] if URL in record else "https://jsonplaceholder.typicode.com/users"
    content_type = extract_content_type_from_uri(url)
    asset = create_asset( # update
        asset_id=asset_id,
        management_url=PROVIDER_MANAGEMENT,
        default_headers=default_headers,
        asset_name=record[TITLE],
        content_type=content_type,
        baseUrl=record[URL]["@id"] if URL in record else "https://jsonplaceholder.typicode.com/users",
        additional_metadata={k: v for k, v in record.items() if k not in [IDENTIFIER, TITLE, URL]},
        proxy=True
    )
    print(f"Asset uploaded with status code: {asset.status_code}")
    contract_definition = create_contract_definition(contract_definition_id, PROVIDER_MANAGEMENT, asset_id, policy_id, default_headers)
    print(f"Contract definition created with status code: {contract_definition.status_code}")

    # rm_asset = delete_asset(asset_id, PROVIDER_MANAGEMENT, default_headers)
    # print(f'Asset removed with code: {rm_asset.status_code}')
    # rm_cdef = delete_contract_definition(contract_definition_id, PROVIDER_MANAGEMENT, default_headers)
    # print(f'Contract definition removed with code: {rm_cdef.status_code}')

    # break
    time.sleep(0.01)
    

In [ ]:
path = '/Users/matt/Downloads/result_keyvalue_fully_simplified.jsonld'
CATALOG_RECORD_TYPE = 'http://www.w3.org/ns/dcat#CatalogRecord'
DATASET_TYPE = 'http://www.w3.org/ns/dcat#Dataset'
PRIMARY_TOPIC = 'http://xmlns.com/foaf/0.1/primaryTopic'
DISTRIBUTION = 'http://www.w3.org/ns/dcat#distribution'
IDENTIFIER = 'http://purl.org/dc/terms/identifier'
TITLE = 'http://purl.org/dc/terms/title'
SPATIAL = 'http://purl.org/dc/terms/spatial'

records : List[Dict] = []
datasets_final = []
datasets_with_distributions = {}
url_to_skip = [
    'http://sdi.iia.cnr.it/gmosgeoserver/ows',
    'https://sdi.iia.cnr.it/gmosgeoserver/ows',
    'https://www.caribic-atmospheric.com/',
    'https://gmos.aeris-data.fr/',
    'https://sdi.iia.cnr.it/hermes/',
    'http://mtcurcio.iia.cnr.it/',
]

def find_in_records_by_id(id : str) -> Dict | None:
    for record in records:
        if '@id' in record and record['@id'] == id:
            return record
    return None

with open(path, 'r') as f:
    records = json.load(f)

for record in records:
    if '@type' in record and DISTRIBUTION in record and DATASET_TYPE in record['@type']:
        datasets_with_distributions[record['@id']] = record

catalogues = [record for record in records if '@type' in record and CATALOG_RECORD_TYPE in record['@type'] and PRIMARY_TOPIC in record]

for catalogue in catalogues:
    if PRIMARY_TOPIC in catalogue:
        datasets_final.append(datasets_with_distributions.get(catalogue[PRIMARY_TOPIC][0]['@id'], None))

datasets_final = [x for x in datasets_final if x is not None]
# Upload policy if not already uploaded
asset_id="Berlin Water Level Daily STA, identifier BerlinWaterLevelDailySTA"
policy_id = "test-policy"
policy = create_policy(policy_id, PROVIDER_MANAGEMENT, default_headers, permissions=[])
print(f'Policy uploaded with status code: {policy.status_code}')

for dataset in datasets_final:
    if SPATIAL in dataset:
        for spatial in dataset[SPATIAL]:
            try:
                res = find_in_records_by_id(spatial['@id'])
            except:
                continue
            if res is not None:
                dataset[SPATIAL] = res
            
    for distribution in dataset[DISTRIBUTION]:
        url : str = unquote(unquote(distribution['@id']))
        if any(skip_url in url for skip_url in url_to_skip):
            print(f"Skipping URL: {url}")
            continue
        name = extract_name_from_uri(url)
        asset_id = f'{extract_name_from_uri(dataset[IDENTIFIER][0])}_{name}'
        contract_definition_id = f"cd_{asset_id}"
        title = f'{dataset[TITLE][0]} - {name}' if TITLE in dataset else "..."
        additional_properties : dict = {k : v for k, v in dataset.items() if k not in [DISTRIBUTION, IDENTIFIER, TITLE, '@type', '@id', DATASET_TYPE]}

        asset = create_asset(
            asset_id=asset_id,
            management_url=PROVIDER_MANAGEMENT,
            default_headers=default_headers,
            asset_name=title,
            baseUrl=url,
            additional_metadata=additional_properties,
            content_type=extract_content_type_from_uri(url),
            proxy=True
        )
        print(f"Asset uploaded with status code: {asset.status_code}")
        contract_definition = create_contract_definition(contract_definition_id, PROVIDER_MANAGEMENT, asset_id, policy_id, default_headers)
        print(f"Contract definition created with status code: {contract_definition.status_code}")

        # rm_asset = delete_asset(asset_id, PROVIDER_MANAGEMENT, default_headers)
        # print(f'Asset removed with code: {rm_asset.status_code}')
        # rm_cdef = delete_contract_definition(contract_definition_id, PROVIDER_MANAGEMENT, default_headers)
        # print(f'Contract definition removed with code: {rm_cdef.status_code}')

        time.sleep(0.01)
    #     break
    # break